In [71]:
# importing necessary libraries
import os
import pandas as pd
import ipywidgets as widgets
import calendar 
from IPython.core.display_functions import clear_output
from IPython.display import display, Markdown
from datetime import datetime, timedelta


# ---- 1. Translations Dictionary ---- #

translations = {
    "project_title": {
        "en": "<div style='text-align: center;'><h1>Car Accident Analysis in Germany</h1><h3><br>An exploratory analysis of car accidents and weather impacts in Germany (2021–2023)</h3></div>",
        "de": "<div style='text-align: center;'><h1>Analyse von Autounfällen in Deutschland</h1><h3><br>Eine explorative Analyse von Verkehrsunfällen und Wettereinflüssen in Deutschland (2021–2023)</h3></div>"
    },

    "summary_heading": {
        "en": "## Project Summary",
        "de": "## Projektzusammenfassung"
    },
    "summary_body": {
        "en":
            """This project aims to conduct a comprehensive data analysis of car accidents in Germany to identify the primary causes and contributing factors. By exploring relevant datasets, we will uncover trends, patterns, and high-risk areas. The insights derived from this analysis will then be used to propose actionable recommendations and improvement strategies for enhancing road safety in Germany, aligning with the "Vision Zero" initiative.
            """,
        "de":
            """Dieses Projekt zielt darauf ab, eine umfassende Datenanalyse von Autounfällen in Deutschland durchzuführen, um die Hauptursachen und beitragenden Faktoren zu identifizieren. Durch die Untersuchung relevanter Datensätze werden wir Trends, Muster und Hochrisikobereiche aufdecken. Die aus dieser Analyse gewonnenen Erkenntnisse werden dann verwendet, um umsetzbare Empfehlungen und Verbesserungsstrategien zur Verbesserung der Verkehrssicherheit in Deutschland vorzuschlagen, im Einklang mit der Initiative "Vision Zero".
            """
    },
    "introduction_heading": {
        "en": "## 1. Introduction and Objectives",
        "de": "## 1. Einführung und Ziele"
    },
    "intro_text_1": {
        "en": """## Objectives
        
 - Identify major causes and contributing factors of car accidents
 - Explore spatial and temporal trends (when and where accidents occur)
 - Assess the role of weather in accident severity
 - Provide recommendations to support safer roads in Germany
""",

        "de": """## Ziele
- Zentrale Ursachen und beitragende Faktoren von Verkehrsunfällen identifizieren
- Räumliche und zeitliche Muster untersuchen (wann und wo Unfälle auftreten)
- Die Rolle des Wetters bei der Unfallschwere bewerten
- Empfehlungen zur Unterstützung sichererer Straßen in Deutschland geben   """
    },
    "introduction_heading_2": {
        "en": "## 2. Data Sources:",
        "de": "## 2. Datenquellen"

    },
    "intro_text_2": {
       "en": """
- Accident data: from Statistische Ämter des Bundes und der Länder (Germany’s official statistics)
Visit [STATISTISCHE ÄMTER](https://unfallatlas.statistikportal.de/) for more information.,
- Weather data: from Open-Meteo API (hourly & daily, 2021–2023)""",
        "de": """
- Unfalldaten: von den Statistischen Ämtern des Bundes und der Länder (offizielle Statistik Deutschlands)
Besuchen Sie uns [STATISTISCHE ÄMTER](https://unfallatlas.statistikportal.de/) für weitere Informationen.
- Wetterdaten: von der Open-Meteo-API (stündlich & täglich, 2021–2023)"""
    },
    "data_decisions": {
    "en": """## 3. Data Decisions

### ⚠️ Why 2020 Was Excluded
During initial exploration, 2020 showed significantly fewer 
accident records than other years.

Investigation revealed this was due to **COVID-19 lockdowns**:
- Strict movement restrictions reduced road traffic dramatically
- Accident patterns in 2020 are not representative of normal conditions
- Including 2020 would skew the analysis results

**Decision:** Analysis covers 2021–2023 only.

### 📊 Weather Data Structure
- **Daily weather** → combined all years → used for main analysis
- **Hourly weather** → kept separate per year (~900MB each)
  → Used for animated weather map
  → Loading all years together would exceed 2.6GB RAM
""",
    "de": """## 3. Datenentscheidungen

### ⚠️ Warum 2020 ausgeschlossen wurde
Bei der ersten Analyse zeigte 2020 deutlich weniger Unfalldaten.

Untersuchung ergab **COVID-19-Lockdowns** als Ursache:
- Bewegungseinschränkungen reduzierten den Straßenverkehr drastisch
- Unfallmuster 2020 nicht repräsentativ für normale Bedingungen
- Einbeziehung würde Ergebnisse verfälschen

**Entscheidung:** Analyse umfasst nur 2021–2023.

### 📊 Wetterdatenstruktur
- **Tägliche Daten** → alle Jahre kombiniert → Hauptanalyse
- **Stündliche Daten** → getrennt pro Jahr (~900MB)
  → Für animierte Wetterkarte
  → Zusammenführung würde 2.6GB RAM überschreiten
"""
},
    "cleaning_prepare": {
        "en": "## 4. Data Cleaning and Preparation process",
        "de": "## 4. Datenbereinigung und Vorbereitungsprozess"
    },
    "loading_file": {
        "en": "### Loading Files and Checking for Correctness",
        "de": " ### Dateien laden und auf Richtigkeit prüfen "
    },
    "check_columns": {
        "en": "### Checking Columns Name",
        "de": "### Spaltennamen prüfen"
    },
    "standardize": {
        "en": " ### Standardize Column Names",
        "de": " ### Spaltennamen standardisieren"
    },
    "table_combined": {
        "en": "### Combining Tables",
        "de": "### Tabellen kombinieren"
    },
    "value_map": {
        "en": " ### Value Mapping and Categorical Transformation",
        "de": " ### Wertezuweisung und Kategorische Transformation"
    },
    "AGS": {
       "en": """ ## 📍 Understanding German Location Codes (AGS)

The accident dataset uses numeric codes instead of city names.
Germany uses an 8-digit official municipality code system:

| Digits | Represents | Example |
|--------|-----------|---------|
| 1-2 | Federal State (Bundesland) | 11 = Berlin |
| 3 | Administrative District | 0 = None |
| 4-5 | District/County | 00 = No division |
| 6-8 | Municipality | 000 = City itself |

Example: 11 0 00 000 = Berlin city

Source: Destatis (German Federal Statistics Office)
We built asg.csv as a lookup table to map these codes
to real city and state names for meaningful analysis. """,
       "de": """ ## 📍 Verständnis der deutschen Standortcodes (AGS)

Der Unfalldatensatz verwendet numerische Codes statt Städtenamen.
Deutschland verwendet ein 8-stelliges Amtliches Gemeindeschlüssel-System:

| Stellen | Bedeutet | Beispiel |
|--------|-----------|---------|
| 1-2 | Bundesland | 11 = Berlin |
| 3 | Regierungsbezirk | 0 = Keine |
| 4-5 | Kreis/Landkreis | 00 = Keine Unterteilung |
| 6-8 | Gemeinde | 000 = Stadt selbst |

Beispiel: 11 0 00 000 = Stadt Berlin

Quelle: Destatis (Statistisches Bundesamt)
Wir haben asg.csv als Nachschlagetabelle aufgebaut, um diese Codes
in echte Städte- und Bundeslandnamen für aussagekräftige Analysen zu übersetzen. """
    },
    "Reorder": {
        "en": "### Reorder Columns",
        "de": "### Spalten neu anordnen"
    },
    "rename": {
        "en": "### Renaming Columns to English for clarity",
        "de": "### Spaltennamen zur besseren Verständlichkeit ins Englische umbenennen"
    },
    "lookup_tables": {
        "en": "### Build a lookup table mapping numeric codes to their descriptions ",
        "de": "### Nachschlagetabelle mit numerischen Codes und deren Bedeutungen erstellen"
    },
    "merge_asg": {
        "en": "### Fill accident data with location names by merging on AGS code components",
        "de": "### Unfalldaten mit Ortsnamen durch Zusammenführen der AGS-Code-Komponenten befüllen"
    },
    "estimated_day": {
        "en": """### 📅 Estimating Missing Dates

The original accident dataset does not include an exact day — 
only **year**, **month**, and **weekday** (e.g., Monday).

To enable merging with hourly weather data, we reconstructed 
an estimated date using the following logic:

1. Take the **1st day of the month**
2. Find the **first occurrence** of the given weekday in that month
3. Combine with the **hour** of accident to create a full datetime

> ⚠️ Note: This is an **estimation** — the exact date is unknown.
> The first occurrence of the weekday in the month is used.
> This may introduce a small margin of error in weather matching.""",
        "de": """### 📅 Schätzung fehlender Datumsangaben

Der ursprüngliche Unfalldatensatz enthält kein genaues Datum —
nur **Jahr**, **Monat** und **Wochentag** (z. B. Montag).

Um eine Zusammenführung mit stündlichen Wetterdaten zu ermöglichen,
haben wir ein geschätztes Datum nach folgender Logik rekonstruiert:

1. Nimm den **1. Tag des Monats**
2. Finde das **erste Vorkommen** des angegebenen Wochentags in diesem Monat
3. Kombiniere mit der **Unfallstunde** zu einem vollständigen Zeitstempel

> ⚠️ Hinweis: Dies ist eine **Schätzung** — das genaue Datum ist unbekannt.
> Das erste Vorkommen des Wochentags im Monat wird verwendet.
> Dies kann zu einer kleinen Fehlerquote beim Wetter-Abgleich führen."""
    },
        "api" : {
        "en": """### 🌐 Prepare data for Open-Meteo API
    Same process applied for all 3 years: 2021, 2022, 2023
    This cell shows the 2023 example — identical logic used for 2021 and 2022""",
        "de": """### 🌐 Daten für die Open-Meteo API vorbereiten
    Derselbe Prozess wurde für alle 3 Jahre angewendet: 2021, 2022, 2023
    Diese Zelle zeigt das 2023-Beispiel — identische Logik für 2021 und 2022"""
    }

}

#---- 2. Language Selector Dropdown ----#

language_selector = widgets.Dropdown(
    options=[('English', 'en'), ('Deutsch', 'de')],
    value='en',  # Default language
    description='Language / Sprache:',
    disabled=False,
    style={'description_width': 'initial'}
)

# ---- 3. Output areas for each translated section ----#

output_area = widgets.Output()  
output_area1 = widgets.Output()
output_area2 = widgets.Output()
output_area3 = widgets.Output()
output_area4 = widgets.Output()
output_area5 = widgets.Output()
output_area6 = widgets.Output()
output_area7 = widgets.Output()
output_area8 = widgets.Output()
output_area9 = widgets.Output()
output_area10 = widgets.Output()
output_area11 = widgets.Output()
output_area12 = widgets.Output()
output_area13 = widgets.Output()


display(language_selector)
display(output_area)
display(output_area1)
display(output_area13)





#---- 4. Rendering Function ----

def render_markdown_content(lang):
    with output_area:  # Direct output to this specific area
        clear_output(wait=True)  # Clear previous content

        # Render each part of your notebook based on the selected language
        display(Markdown(translations["project_title"][lang]))
        display(Markdown(translations["summary_heading"][lang]))
        display(Markdown(translations["summary_body"][lang]))

        display(Markdown("---"))  # Separator

        display(Markdown(translations["introduction_heading"][lang]))
        display(Markdown(translations["intro_text_1"][lang]))
        display(Markdown(translations["introduction_heading_2"][lang]))
        display(Markdown(translations["intro_text_2"][lang]))
        display(Markdown(translations["data_decisions"][lang]))
        display(Markdown(translations["cleaning_prepare"][lang]))

    with output_area1:
        clear_output(wait=True)
        display(Markdown(translations["loading_file"][lang]))

    with output_area2:
        clear_output(wait=True)
        display(Markdown(translations["check_columns"][lang]))

    with output_area3:
        clear_output(wait=True)
        display(Markdown(translations["standardize"][lang]))

    with output_area4:
        clear_output(wait=True)
        display(Markdown(translations["table_combined"][lang]))

    with output_area5:
        clear_output(wait=True)
        display(Markdown(translations["value_map"][lang]))

    with output_area6:
        clear_output(wait=True)
        display(Markdown(translations["AGS"][lang]))

    with output_area7:
        clear_output(wait=True)
        display(Markdown(translations["Reorder"][lang]))

    with output_area8:
        clear_output(wait=True)
        display(Markdown(translations["rename"][lang]))

    with output_area9:
        clear_output(wait=True)
        display(Markdown(translations["lookup_tables"][lang]))

    with output_area10:
        clear_output(wait=True)
        display(Markdown(translations["merge_asg"][lang]))

    with output_area11:
        clear_output(wait=True)
        display(Markdown(translations["estimated_day"][lang]))

    with output_area12:
        clear_output(wait=True)
        display(Markdown(translations["api"][lang]))

        



# ---- 5. Observe language changes ----

# Call render_markdown_content whenever the dropdown value changes
language_selector.observe(lambda change: render_markdown_content(change.new), names='value')


# Initial render of the content
render_markdown_content(language_selector.value)

Dropdown(description='Language / Sprache:', options=(('English', 'en'), ('Deutsch', 'de')), style=DescriptionS…

Output()

Output()

Output()

In [72]:
# Base directory
BASE_DIR = os.path.dirname(os.path.abspath('__file__'))

# Accident data
df_2021 = pd.read_csv(os.path.join(BASE_DIR, 'data', 'accidents_2021.csv'), low_memory=False)
df_2022 = pd.read_csv(os.path.join(BASE_DIR, 'data', 'accidents_2022.csv'), sep=";", low_memory=False)
df_2023 = pd.read_csv(os.path.join(BASE_DIR, 'data', 'accidents_2023.csv'), sep=";", low_memory=False)

# Lookup table
off_code = pd.read_csv(os.path.join(BASE_DIR, 'asg.csv'), dtype=str)

In [73]:
display(output_area2)

Output()

In [74]:
print("2021")
print(df_2021.columns.to_list())
print("2022")
print(df_2022.columns.to_list())
print("2023")
print(df_2023.columns.to_list())

2021
['ID', 'ULAND', 'UREGBEZ', 'UKREIS', 'UGEMEINDE', 'UJAHR', 'UMONAT', 'USTUNDE', 'UWOCHENTAG', 'UKATEGORIE', 'UART', 'UTYP1', 'IstRad', 'IstPKW', 'IstFuss', 'IstKrad', 'IstSonstige', 'ULICHTVERH', 'IstGkfz', 'LINREFX', 'LINREFY', 'XGCSWGS84', 'YGCSWGS84', 'UIDENTSTLAE', 'IstStrassenzustand']
2022
['OBJECTID', 'UIDENTSTLAE', 'ULAND', 'UREGBEZ', 'UKREIS', 'UGEMEINDE', 'UJAHR', 'UMONAT', 'USTUNDE', 'UWOCHENTAG', 'UKATEGORIE', 'UART', 'UTYP1', 'ULICHTVERH', 'IstStrassenzustand', 'IstRad', 'IstPKW', 'IstFuss', 'IstKrad', 'IstGkfz', 'IstSonstige', 'LINREFX', 'LINREFY', 'XGCSWGS84', 'YGCSWGS84']
2023
['OID_', 'UIDENTSTLAE', 'ULAND', 'UREGBEZ', 'UKREIS', 'UGEMEINDE', 'UJAHR', 'UMONAT', 'USTUNDE', 'UWOCHENTAG', 'UKATEGORIE', 'UART', 'UTYP1', 'ULICHTVERH', 'IstStrassenzustand', 'IstRad', 'IstPKW', 'IstFuss', 'IstKrad', 'IstGkfz', 'IstSonstige', 'LINREFX', 'LINREFY', 'XGCSWGS84', 'YGCSWGS84', 'PLST']


In [75]:
display(output_area3)

Output()

In [76]:
def standardize_columns(df):
    rename_dict = {'OBJECTID': 'id',
                   'OID_': 'id',
                   'ID': 'id'}

    df = df.rename(columns=rename_dict)
    return df


df_2021 = df_2021.pipe(standardize_columns)
df_2022 = df_2022.pipe(standardize_columns)
df_2023 = df_2023.pipe(standardize_columns)


In [77]:
display(output_area4)

Output()

In [78]:
all_columns = set(df_2021.columns) | set(df_2022.columns) | set(df_2023.columns)

In [79]:
print(all_columns)

{'UTYP1', 'PLST', 'YGCSWGS84', 'IstSonstige', 'LINREFX', 'IstPKW', 'UREGBEZ', 'USTUNDE', 'UJAHR', 'UKREIS', 'UKATEGORIE', 'IstGkfz', 'IstKrad', 'IstStrassenzustand', 'IstRad', 'id', 'ULAND', 'UART', 'XGCSWGS84', 'UWOCHENTAG', 'LINREFY', 'UIDENTSTLAE', 'UGEMEINDE', 'ULICHTVERH', 'IstFuss', 'UMONAT'}


In [80]:
merged_df = pd.concat([df_2021, df_2022, df_2023], ignore_index=True)
merged_df = merged_df.set_index("id")

merged_df.head()

,ULAND,UREGBEZ,UKREIS,UGEMEINDE,UJAHR,UMONAT,USTUNDE,UWOCHENTAG,UKATEGORIE,UART,...,IstSonstige,ULICHTVERH,IstGkfz,LINREFX,LINREFY,XGCSWGS84,YGCSWGS84,UIDENTSTLAE,IstStrassenzustand,PLST
id,,,,,,,,,,,,,,,,,,,,,
1,1,0,54,165,2021,3,7,2,2,8,...,0,0,0,"483995,394383990205824","6069091,089339181780815","8,751233225000021","54,768786524000063",01210308125013512021,2,NaN
2,1,0,2,0,2021,6,15,3,3,2,...,0,0,0,"573010,098060709424317","6020090,876354970037937","10,122558347000052","54,323450168000079",01210608134013112021,0,NaN
3,1,0,61,7,2021,6,13,5,3,8,...,1,0,0,"527231,388501240871847","5972658,255789350718260","9,414456907000044","53,901644518000069",01210610181013902021,0,NaN
4,1,0,53,41,2021,5,11,2,3,0,...,0,0,0,"614902,973624786362052","5963896,514101063832641","10,745101073000058","53,810912932000065",01210524161013132021,0,NaN
5,1,0,55,32,2021,5,15,7,3,5,...,0,0,0,"617331,180263186804950","5996138,282804258167744","10,794357030000072","54,100018150000039",01210529152013382022,0,NaN


In [81]:
merged_df.shape

(764366, 25)

In [82]:
# Add note about Amtlicher Gemeindeschlüssel (AGS)
merged_df.columns

Index(['ULAND', 'UREGBEZ', 'UKREIS', 'UGEMEINDE', 'UJAHR', 'UMONAT', 'USTUNDE',
       'UWOCHENTAG', 'UKATEGORIE', 'UART', 'UTYP1', 'IstRad', 'IstPKW',
       'IstFuss', 'IstKrad', 'IstSonstige', 'ULICHTVERH', 'IstGkfz', 'LINREFX',
       'LINREFY', 'XGCSWGS84', 'YGCSWGS84', 'UIDENTSTLAE',
       'IstStrassenzustand', 'PLST'],
      dtype='str')

In [83]:
display(output_area6)

Output()

In [84]:
# normalize AGS code
merged_df['ULAND'] = merged_df['ULAND'].astype(str).str.strip().str.zfill(2)
merged_df['UKREIS'] = merged_df['UKREIS'].astype(str).str.strip().str.zfill(2)
merged_df['UGEMEINDE'] = merged_df['UGEMEINDE'].astype(str).str.strip().str.zfill(3)
merged_df['UREGBEZ'] = merged_df['UREGBEZ'].astype(str).str.strip().str.zfill(1)

# also accident_uid
merged_df['UIDENTSTLAE'] = merged_df['UIDENTSTLAE'].astype(str).str.strip()

# add new column for full code
merged_df['ASG'] = (merged_df['ULAND']
                    + merged_df['UREGBEZ']
                    + merged_df['UKREIS']
                    + merged_df['UGEMEINDE'])

#Change type for 'LINREFX', 'LINREFY', 'XGCSWGS84', 'YGCSWGS84', change seperator to (.) as European separator
merged_df['LINREFX'] = merged_df['LINREFX'].str.replace(',', '.', regex=False).astype(float)
merged_df['LINREFY'] = merged_df['LINREFY'].str.replace(',', '.', regex=False).astype(float)
merged_df['XGCSWGS84'] = merged_df['XGCSWGS84'].str.replace(',', '.', regex=False).astype(float)
merged_df['YGCSWGS84'] = merged_df['YGCSWGS84'].str.replace(',', '.', regex=False).astype(float)

In [85]:
display(output_area7)

Output()

In [86]:
merged_df = merged_df[['ULAND', 'UREGBEZ', 'UKREIS', 'UGEMEINDE', 'ASG','UIDENTSTLAE',
                       'UJAHR', 'UMONAT', 'USTUNDE', 'UWOCHENTAG',
                       'UKATEGORIE', 'UART', 'UTYP1', 'ULICHTVERH', 'IstStrassenzustand',
                       'IstGkfz', 'IstPKW', 'IstKrad', 'IstRad', 'IstFuss', 'IstSonstige',
                       'LINREFX', 'LINREFY', 'XGCSWGS84', 'YGCSWGS84', 'PLST']]

In [87]:
merged_df.sample(5)

,ULAND,UREGBEZ,UKREIS,UGEMEINDE,ASG,UIDENTSTLAE,UJAHR,UMONAT,USTUNDE,UWOCHENTAG,...,IstPKW,IstKrad,IstRad,IstFuss,IstSonstige,LINREFX,LINREFY,XGCSWGS84,YGCSWGS84,PLST
id,,,,,,,,,,,,,,,,,,,,,
107840,09,1,71,132,09171132,09220724001571293390,2022,7,10,1,...,1,0,0,0,1,767829.950553,5.351994e+06,12.609183,48.264464,NaN
248501,12,0,65,136,12065136,12230905442505470710,2023,9,17,3,...,0,0,1,0,0,784030.843000,5.839070e+06,13.197223,52.627153,1.0
248214,12,0,67,144,12067144,12230926522107518570,2023,9,10,3,...,1,0,0,1,0,845520.588900,5.813985e+06,14.076320,52.366924,1.0
221419,12,0,54,000,12054000,12210730732101930850,2021,7,21,6,...,1,0,1,0,0,774121.827733,5.812086e+06,13.028972,52.390259,NaN
232633,11,0,01,001,11001001,11220528110000083641,2022,5,19,7,...,1,0,1,0,0,797396.798672,5.828684e+06,13.384804,52.526908,NaN


In [88]:
display(output_area8)

Output()

In [89]:
merged_df.rename(columns={
                          'ULAND': 'state',
                          'UREGBEZ': 'government_district',
                          'UKREIS': 'district',
                          'UGEMEINDE': 'municipality',
                          'ASG': 'official_admin_code',
                          'UIDENTSTLAE': 'accident_uid',
                          'UJAHR': 'year',
                          'UMONAT': 'month',
                          'USTUNDE': 'hour',
                          'UWOCHENTAG': 'day',
                          'UKATEGORIE': 'category',
                          'ULICHTVERH': 'light',
                          'IstGkfz': 'goods_vehicle_involved',
                          'IstKrad': 'motorcycle_involved',
                          'IstSonstige': 'other_involved',
                          'UART': 'kind',
                          'UTYP1': 'type',
                          'IstRad': 'bike_involved',
                          'IstPKW': 'passenger_car_involved',
                          'IstFuss': 'passenger_involved',
                          'LINREFX': 'utm_easting',
                          'LINREFY': 'utm_northing',
                          'XGCSWGS84': 'longitude',
                          'YGCSWGS84': 'latitude',
                          'IstStrassenzustand': 'road_surface'}, inplace=True)

In [90]:
# Missing value handling
merged_df.isnull().sum()

state                          0
government_district            0
district                       0
municipality                   0
official_admin_code            0
accident_uid                   0
year                           0
month                          0
hour                           0
day                            0
category                       0
kind                           0
type                           0
light                          0
road_surface                   0
goods_vehicle_involved         0
passenger_car_involved         0
motorcycle_involved            0
bike_involved                  0
passenger_involved             0
other_involved                 0
utm_easting                    0
utm_northing                   0
longitude                      0
latitude                       0
PLST                      495318
dtype: int64

In [91]:
#Drop column PLST ( unnecessary column)
merged_df.drop(columns=['PLST'], inplace=True)

In [92]:
merged_df.columns

Index(['state', 'government_district', 'district', 'municipality',
       'official_admin_code', 'accident_uid', 'year', 'month', 'hour', 'day',
       'category', 'kind', 'type', 'light', 'road_surface',
       'goods_vehicle_involved', 'passenger_car_involved',
       'motorcycle_involved', 'bike_involved', 'passenger_involved',
       'other_involved', 'utm_easting', 'utm_northing', 'longitude',
       'latitude'],
      dtype='str')

In [93]:
merged_df.info()

<class 'pandas.DataFrame'>
Index: 764366 entries, 1 to 269048
Data columns (total 25 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   state                   764366 non-null  str    
 1   government_district     764366 non-null  str    
 2   district                764366 non-null  str    
 3   municipality            764366 non-null  str    
 4   official_admin_code     764366 non-null  str    
 5   accident_uid            764366 non-null  str    
 6   year                    764366 non-null  int64  
 7   month                   764366 non-null  int64  
 8   hour                    764366 non-null  int64  
 9   day                     764366 non-null  int64  
 10  category                764366 non-null  int64  
 11  kind                    764366 non-null  int64  
 12  type                    764366 non-null  int64  
 13  light                   764366 non-null  int64  
 14  road_surface            764366 non-n

In [94]:
merged_df.describe()

,year,month,hour,day,category,kind,type,light,road_surface,goods_vehicle_involved,passenger_car_involved,motorcycle_involved,bike_involved,passenger_involved,other_involved,utm_easting,utm_northing,longitude,latitude
count,764366.000000,764366.000000,764366.000000,764366.000000,764366.000000,764366.000000,764366.000000,764366.000000,764366.000000,764366.000000,764366.000000,764366.000000,764366.000000,764366.000000,764366.000000,764366.000000,7.643660e+05,764366.000000,764366.000000
mean,2022.039539,6.846020,13.360903,4.099832,2.806839,3.815130,3.817245,0.429607,0.283207,0.046153,0.748619,0.132157,0.322552,0.087715,0.120792,550630.366128,5.638272e+06,9.722854,50.873742
std,0.814172,3.164258,4.794298,1.874647,0.418024,2.666503,2.174978,0.788050,0.500269,0.209817,0.433807,0.338661,0.467453,0.282879,0.325885,150838.517687,1.986606e+05,2.156931,1.783071
min,2021.000000,1.000000,0.000000,1.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,280699.214691,5.241009e+06,5.871498,47.315548
25%,2021.000000,4.000000,10.000000,3.000000,3.000000,2.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,427236.705605,5.471575e+06,7.961419,49.385249
50%,2022.000000,7.000000,14.000000,4.000000,3.000000,4.000000,3.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,534090.214298,5.660068e+06,9.487300,51.039263
75%,2023.000000,9.000000,17.000000,6.000000,3.000000,5.000000,6.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,0.000000,676714.766727,5.794902e+06,11.438925,52.289936
max,2023.000000,12.000000,23.000000,7.000000,3.000000,9.000000,7.000000,2.000000,2.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,920405.489500,6.100725e+06,15.028387,55.051843


In [95]:
display(output_area5)

Output()

In [96]:
# Replace weekdays codes
weekday_map = {1: 'sunday',
               2: 'monday',
               3: 'tuesday',
               4: 'wednesday',
               5: 'thursday',
               6: 'friday',
               7: 'saturday'}
merged_df['day'] = merged_df['day'].map(weekday_map)

In [97]:
# Replace accident category codes
category_map = {1: 'killed',
                2: 'seriously injury ',
                3: 'slightly injury'}
merged_df['category'] = merged_df['category'].map(category_map)

In [98]:
# Repalce land codes
land_map = {'01': 'schleswig-holstein',
            '02': 'hamburg',
            '03': 'niedersachsen',
            '04': 'bremen',
            '05': 'nordrhein-westfalen',
            '06': 'hessen',
            '07': 'rheinland-pfalz',
            '08': 'baden-württemberg',
            '09': 'bayern',
            '10': 'saarland',
            '11': 'berlin',
            '12': 'brandenburg',
            '13': 'mecklenburg-vorpommern',
            '14': 'sachsen',
            '15': 'sachsen-anhalt',
            '16': 'thüringen'}
merged_df['state'] = merged_df['state'].map(land_map)

In [99]:
# Replace month 
month_map = {1: "January",
             2: "February",
             3: "March",
             4: "April",
             5: "May",
             6: "June",
             7: "July",
             8: "August",
             9: "September",
             10: "October",
             11: "November",
             12: "December"}
merged_df['month'] = merged_df['month'].map(month_map)

In [100]:
#Replace light code
light_map = {0: 'daylight',
             1: 'twilight',
             2: 'darkness'}
merged_df['light'] = merged_df['light'].map(light_map)

In [101]:
#Replace bike code
bike_map = {0: 'acc_nobike',
            1: 'acc_bike'}
merged_df['bike_involved'] = merged_df['bike_involved'].map(bike_map)

In [102]:
#Replace passenger_car_involved code
pcar_map = {0: 'no_car',
            1: 'car'}
merged_df['passenger_car_involved'] = merged_df['passenger_car_involved'].map(pcar_map)

In [103]:
#Replace passenger_involved code
pass_map = {0: 'no_pass',
            1: 'pass'}
merged_df['passenger_involved'] = merged_df['passenger_involved'].map(pass_map)

In [104]:
#Replace passenger_involved code
moto_map = {0: 'no_moto',
            1: 'moto'}
merged_df['motorcycle_involved'] = merged_df['motorcycle_involved'].map(moto_map)

In [105]:
#Replace goods_vehicle_involved code
goods_map = {0: 'no_goods',
             1: 'goods'}
merged_df['goods_vehicle_involved'] = merged_df['goods_vehicle_involved'].map(goods_map)

In [106]:
#Replace other_involved code
other_map = {0: 'no_other',
             1: 'other'}
merged_df['other_involved'] = merged_df['other_involved'].map(other_map)

In [107]:
#Replace road_surface code
surface_map = {0: 'dry',
               1: 'wet/damp',
               2: 'slippery(winter)'}
merged_df['road_surface'] = merged_df['road_surface'].map(surface_map)

In [108]:
merged_df.head()

,state,government_district,district,municipality,official_admin_code,accident_uid,year,month,hour,day,...,goods_vehicle_involved,passenger_car_involved,motorcycle_involved,bike_involved,passenger_involved,other_involved,utm_easting,utm_northing,longitude,latitude
id,,,,,,,,,,,,,,,,,,,,,
1,schleswig-holstein,0,54,165,01054165,01210308125013512021,2021,March,7,monday,...,no_goods,car,no_moto,acc_nobike,no_pass,no_other,483995.394384,6.069091e+06,8.751233,54.768787
2,schleswig-holstein,0,02,000,01002000,01210608134013112021,2021,June,15,tuesday,...,no_goods,car,no_moto,acc_nobike,no_pass,no_other,573010.098061,6.020091e+06,10.122558,54.323450
3,schleswig-holstein,0,61,007,01061007,01210610181013902021,2021,June,13,thursday,...,no_goods,no_car,no_moto,acc_bike,no_pass,other,527231.388501,5.972658e+06,9.414457,53.901645
4,schleswig-holstein,0,53,041,01053041,01210524161013132021,2021,May,11,monday,...,no_goods,no_car,no_moto,acc_bike,no_pass,no_other,614902.973625,5.963897e+06,10.745101,53.810913
5,schleswig-holstein,0,55,032,01055032,01210529152013382022,2021,May,15,saturday,...,no_goods,car,no_moto,acc_bike,no_pass,no_other,617331.180263,5.996138e+06,10.794357,54.100018


In [109]:
display(output_area9)

Output()

In [110]:
df = pd.DataFrame({'acc_kind_code': [1, 2, 3, 4, 5, 6, 7, 8, 9, 0]})

# Lookup table
lookup = {1: 'Collision with another vehicle which starts, stops or is stationary',
          2: 'Collision with another vehicle moving ahead or waiting',
          3: 'Collision with another vehicle moving laterally in the same direction',
          4: 'Collision with another oncoming vehicle',
          5: 'Collision with another vehicle which turns into or crosses a road',
          6: 'Collision between vehicle and pedestrian',
          7: 'Collision with an obstacle in the carriageway',
          8: 'Leaving the carriageway to the right',
          9: 'Leaving the carriageway to the left',
          0: 'Accident of another kind'}

# map the full description
df['kind_desc'] = df['acc_kind_code'].map(lookup)

# print(df)

In [111]:
df = pd.DataFrame({'acc_type_code': [1, 2, 3, 4, 5, 6, 7]})
#Look table
lookup = {1: 'Driving accident',
          2: 'Accident caused by turning off the road',
          3: 'Accident caused by turning into a road or by crossing it',
          4: 'Accident caused by crossing the road',
          5: 'Accident involving stationary',
          6: 'Accident between vehicles moving along in carriageway',
          7: 'Other accident'}
# map full description
df['type_desc'] = df['acc_type_code'].map(lookup)
# print(df)

In [112]:
# for off_code fill na , lower case
off_code = off_code.fillna('   ')
cols_to_lower = ['state', 's_district', 's_municipality']

for col in cols_to_lower:
    off_code[col] = off_code[col].str.lower()

In [113]:
off_code.head()

,state,s_code,code,s_district,s_municipality
0,schleswig-holstein,01,,schleswig-holstein,kiel
1,schleswig-holstein,01001,,"flensburg, stadt",flensburg
2,schleswig-holstein,010010000000,01001000,"flensburg, stadt",flensburg
3,schleswig-holstein,01002,,"kiel, landeshauptstadt",kiel
4,schleswig-holstein,010020000000,01002000,"kiel, landeshauptstadt",kiel


In [114]:
# check duplicates
off_code['code'].duplicated().count()

np.int64(13386)

In [115]:
display(output_area10)

Output()

In [116]:
# Merge on the matching code for column municipality

#remove duplication
clean_muni = off_code[['code', 's_municipality']].drop_duplicates()

# merge
merged_df = merged_df.merge(clean_muni, left_on='official_admin_code', right_on='code', how='left')

# Fill municipality with s_municipality from off_code
merged_df['municipality'] = merged_df['s_municipality']

#clean
merged_df.drop(columns=['code', 's_municipality'], inplace=True)

In [117]:
# Merge on the matching code for column district 

# extract first 5 num from official admin code
merged_df['code_5'] = merged_df['official_admin_code'].str[:5]

#remove duplication from 's_code','s_district'

clean_district = off_code[['s_code', 's_district']].drop_duplicates()

# merge
merged_df = merged_df.merge(clean_district, left_on='code_5', right_on='s_code', how='left')

# Fill municipality with s_municipality from off_code
merged_df['district'] = merged_df['s_district']

#clean
merged_df.drop(columns=['code_5', 's_code', 's_district'], inplace=True)

In [118]:
# Merge on the matching code for column government_district

# extract first 3 num from official admin code
merged_df['code_3'] = merged_df['official_admin_code'].str[:3]

#remove duplication from 's_code','s_district'

look_gov = off_code[['s_code', 's_district']].drop_duplicates()

# merge
merged_df = merged_df.merge(look_gov, left_on='code_3', right_on='s_code', how='left')

# Fill municipality with s_municipality from off_code
merged_df['government_district'] = merged_df['s_district'].fillna('none')

#clean
merged_df.drop(columns=['code_3', 's_code', 's_district'], inplace=True)

In [119]:
# Search for States have NAN Value
states_with_nan = merged_df.loc[merged_df['municipality'].isna(), 'state'].unique()
states_with_nan

<ArrowStringArray>
[      'hamburg', 'niedersachsen',        'hessen',        'bayern',
        'berlin',   'brandenburg',       'sachsen',     'thüringen']
Length: 8, dtype: str

In [120]:
# Filter the merged_df for rows where state is 'sachsen' and municipality is NaN
target_state = 'sachsen'

# Define the conditions
condition_state = merged_df['state'] == target_state
condition_district_nan = merged_df['municipality'].isna()

# Combine the conditions using the logical AND operator (&)
s = merged_df[condition_state & condition_district_nan]
s

,state,government_district,district,municipality,official_admin_code,accident_uid,year,month,hour,day,...,goods_vehicle_involved,passenger_car_involved,motorcycle_involved,bike_involved,passenger_involved,other_involved,utm_easting,utm_northing,longitude,latitude
125595,sachsen,none,mittelsachsen,NaN,14522450,14210217114200150990,2021,February,12,wednesday,...,goods,car,no_moto,acc_nobike,no_pass,no_other,790237.548309,5.682352e+06,13.156576,51.218638
125885,sachsen,none,mittelsachsen,NaN,14522450,14210427119310320370,2021,April,12,tuesday,...,goods,no_car,no_moto,acc_nobike,no_pass,other,789621.021000,5.678693e+06,13.144820,51.186136
126021,sachsen,none,mittelsachsen,NaN,14522450,14210303119320185250,2021,March,21,wednesday,...,no_goods,no_car,moto,acc_nobike,no_pass,no_other,785378.127076,5.679988e+06,13.085294,51.199883
127121,sachsen,none,mittelsachsen,NaN,14522450,14210921114200718050,2021,September,9,tuesday,...,goods,car,no_moto,acc_nobike,no_pass,no_other,790123.151110,5.680785e+06,13.153675,51.204646
127493,sachsen,none,mittelsachsen,NaN,14522450,14210728119310569910,2021,July,17,wednesday,...,no_goods,car,no_moto,acc_nobike,no_pass,no_other,789524.747001,5.677960e+06,13.142855,51.179607
127574,sachsen,none,mittelsachsen,NaN,14522450,14210716114200521170,2021,July,14,friday,...,no_goods,no_car,no_moto,acc_nobike,pass,other,790734.661258,5.680395e+06,13.162090,51.200838
128001,sachsen,none,mittelsachsen,NaN,14522620,14210616114200436380,2021,June,15,wednesday,...,goods,car,no_moto,acc_nobike,no_pass,no_other,789394.226262,5.676834e+06,13.140087,51.169579
128305,sachsen,none,mittelsachsen,NaN,14522620,14210923119310738450,2021,September,16,thursday,...,no_goods,car,no_moto,acc_nobike,no_pass,no_other,792880.319287,5.678657e+06,13.191302,51.184155
128446,sachsen,none,mittelsachsen,NaN,14522450,14210916119320705140,2021,September,6,thursday,...,no_goods,car,no_moto,acc_nobike,pass,no_other,788423.288092,5.681167e+06,13.129713,51.208932
129545,sachsen,none,mittelsachsen,NaN,14522450,14211129119310038240,2021,November,11,monday,...,goods,car,no_moto,acc_nobike,no_pass,no_other,789626.305571,5.678845e+06,13.145018,51.187498


In [121]:
#handel missing data for Berlin and Hamburg the district should be (berlin city), also municipality should be berlin and for hamburg(Hamburg, Freie und Hansestadt

value = ['district', 'municipality']
#handle berlin
berlin_nan = merged_df['state'] == 'berlin'
merged_df.loc[berlin_nan, value] = 'berlin'

#handle value
hamburg_nan = merged_df['state'] == 'hamburg'
merged_df.loc[hamburg_nan, value] = 'hamburg'

In [122]:
filtered_df = merged_df[merged_df['state'] == 'berlin']
# filtered_df

In [123]:
# Convert 'month' from text to number
merged_df['month_num'] = pd.to_datetime(merged_df['month'], format='%B').dt.month

In [124]:
display(output_area11)

Output()

In [125]:
#Estimate Day Based on year,month, weekday
#Reconstruct exact date from year, month, and weekday
def guess_day(row):
    target_weekday = row['day'].capitalize()
    year = row['year']
    month = row['month_num']

    # Start from 1st of month
    first_day = datetime(year, month, 1)
    days_to_add = (list(calendar.day_name).index(target_weekday) - first_day.weekday()) % 7
    return first_day + timedelta(days=days_to_add)


# Apply this to your DataFrame

merged_df['estimated_date'] = merged_df.apply(guess_day, axis=1)

# Now combine with hour
merged_df['datetime'] = pd.to_datetime(merged_df['estimated_date']) + pd.to_timedelta(merged_df['hour'], unit='h')

In [126]:
# Save as clean_car_dataset
merged_df.to_csv('cleaned_car_dataset.csv', index=False)

In [127]:
display(output_area12)


Output()

In [ ]:
#Prepare data for open meteo API - Year 2023
# Step 1: Convert to datetime and keep only valid dates
merged_df['date'] = pd.to_datetime(merged_df['estimated_date'], errors='coerce')
df = merged_df.dropna(subset=['date'])

# Step 2: Filter for 2023 only
meteo_2023 = df[df['date'].dt.year == 2023].copy()

# Step 3: Drop invalid or missing coordinates
meteo_2023 = meteo_2023.dropna(subset=['latitude', 'longitude']).copy()
meteo_2023 = meteo_2023[
    (meteo_2023['latitude'].between(-90, 90)) &
    (meteo_2023['longitude'].between(-180, 180)) &
    (meteo_2023['latitude'] != 0) & (meteo_2023['longitude'] != 0)
].copy()

# Step 4: Format date as string
meteo_2023['date_str'] = meteo_2023['date'].dt.strftime("%Y-%m-%d")

meteo_2023 = meteo_2023.drop_duplicates(subset=['latitude', 'longitude', 'date_str'])

# Step 5: Create Open-Meteo format string: lat,lon,,Europe/Berlin,date,date
meteo_2023['location_format'] = meteo_2023.apply(
    lambda row: f"{row['latitude']},{row['longitude']},,Europe/Berlin,{row['date_str']},{row['date_str']}",
    axis=1
)

# Step 6: Export to one .txt file for Open-Meteo
output_file = "openmeteo_all_germany_2023.txt"

# Write lines manually
with open(output_file, 'w') as f:
    for line in meteo_2023['location_format']:
        f.write(line + '\n') # Add a newline character after each line
print(f"✅ Saved {len(meteo_2023)} rows to {output_file}")

✅ Saved 268916 rows to openmeteo_all_germany_2023.txt
